In [ ]:
!pip install xgboost lightgbm catboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 1.3 MB/s eta 0:01:19
   ---------------------------------------- 0.8/101.7 MB 1.4 MB/s eta 0:01:15
   ---------------------------------------- 0.8/101.7 MB 1.4 MB/s eta 0:01:15
    --------------------------------------- 1.3/101.7 MB 1.3 MB/s eta 0:01:18
    --------------------------------------- 1.6/101.7 MB 1.4 MB/s eta 0:01:13
    --------------------------------------- 1.8/101.7 MB 1.4 MB/s eta 0:01:14
    --------------------------------------- 2.1/101.7 MB 1.3 MB/s eta 0:01:17
   - -------------------------------------- 2.6/101.7 MB 1.4 MB/s eta 0:01:11
   - -------------------------------------- 2.9/101.7 MB 1.5 MB/s eta 0:01:09
   - -------------------------------------- 3.4/101.7 MB 1.5 MB/s eta 0:01:05
   - -------------------------------------- 3.9/101.7 MB 1.6 MB/s eta 0:01:01



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from category_encoders import TargetEncoder

import numpy as np
import pandas as pd

In [ ]:
df=pd.read_csv('data/gandhinagar_property_apartments_final.csv')
df.head()

,Price,Area,Bathrooms,Balconies,Current_Floor,Total_Floors,Furnishing_Status,Mapped_Area,Facing,Property_Age,Bedrooms,Area_Type,Property_Status
0,126.00,2916.0,3.0,1.0,4.0,8.0,Unknown,Sargasan,Unknown,New,3.0,Super Built-up,Ready_to_Move
1,51.99,1755.0,3.0,2.0,3.0,7.0,Unknown,Pethapur,Unknown,New,3.0,Super Built-up,Under_Construction
2,97.00,1908.0,3.0,2.0,4.0,8.0,Unknown,Raysan,Unknown,New,3.0,Super Built-up,Ready_to_Move
3,125.00,2205.0,2.0,2.0,9.0,13.0,Unknown,Raysan,Unknown,New,3.0,Carpet,Ready_to_Move
4,79.00,1755.0,3.0,1.0,8.0,13.0,unfurnished,Randesan,Unknown,5-10 years,3.0,Carpet,Ready_to_Move


In [49]:
X=df.drop(columns=['Price'])
y=df['Price']

In [50]:
y_transformed = np.log1p(y)

In [ ]:
def scorer(model_name, model):

    output = []

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])


    # Cross Validation R²
    kfold = KFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    cv_r2 = cross_val_score(
        pipeline,
        X,
        y_transformed,
        cv=kfold,
        scoring='r2'
    ).mean()

    # Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_transformed,
        test_size=0.2,
        random_state=42
    )

    pipeline.fit(X_train, y_train)

    y_pred_log = pipeline.predict(X_test)

    # Test R² on log scale
    test_r2 = r2_score(y_test, y_pred_log)

    # Convert back to original scale
    y_test_original = np.expm1(y_test)
    y_pred_original = np.expm1(y_pred_log)

    mae = mean_absolute_error(
        y_test_original,
        y_pred_original
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test_original,
            y_pred_original
        )
    )

    output.extend([
        model_name,
        cv_r2,
        test_r2,
        mae,
        rmse
    ])

    return output

### Ordinal Encoding

In [73]:
columns_to_encode=['Mapped_Area','Furnishing_Status','Facing','Property_Age','Area_Type','Property_Status']

In [79]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Area', 'Bathrooms', 'Balconies', 'Current_Floor', 'Total_Floors', 'Bedrooms']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [92]:
model_dict = {

    # Linear Models
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),

    # Support Vector
    'SVR': SVR(),

    # Tree Models
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42),

    # Boosting
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'AdaBoost': AdaBoostRegressor(random_state=42),
    'XGBoost': XGBRegressor(
        random_state=42,
        verbosity=0
    ),
    'LightGBM': LGBMRegressor(
        random_state=42,
        verbose=-1
    ),
    'CatBoost': CatBoostRegressor(
        random_state=42,
        verbose=0
    ),

    # Neural Network
    'MLP': MLPRegressor(
        random_state=42,
        max_iter=1000
    )
}

In [93]:
model_output = []

for model_name, model in model_dict.items():
    model_output.append(
        scorer(model_name, model)
    )

c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\MEGH BAVARVA\Desktop\Da

In [ ]:
model_df = pd.DataFrame(
    model_output,
    columns=[
        'Model',
        'CV_R2',
        'Test_R2',
        'MAE' ,
        'RMSE'
    ]
)

model_df.sort_values(
    by='MAE',
    ascending=True
).reset_index(drop=True)

,Model,CV_R2,Test_R2,MAE,RMSE
0,Extra Trees,0.835510,0.852689,15.477647,24.484725
1,Random Forest,0.840400,0.851957,15.829412,25.438885
2,Gradient Boosting,0.840182,0.845942,15.862121,25.605352
3,CatBoost,0.852181,0.844661,15.901018,25.933712
4,LightGBM,0.837811,0.842135,16.841208,27.691557
5,XGBoost,0.829533,0.824257,17.140123,27.792981
6,SVR,0.797595,0.822018,17.456374,28.228738
7,Decision Tree,0.727436,0.735414,19.093607,32.973021
8,Ridge,0.727616,0.779277,20.941545,34.693636
9,Linear Regression,0.727573,0.779230,20.959509,34.738970


### One-Hot Encoding

In [104]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Area', 'Bathrooms', 'Balconies', 'Current_Floor', 'Total_Floors', 'Bedrooms']),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'),
         ['Mapped_Area','Property_Age','Furnishing_Status','Area_Type','Property_Status','Facing'])
    ], 
    remainder='passthrough'
)

In [100]:
model_dict = {

    # Linear Models
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),

    # Support Vector
    'SVR': SVR(),

    # Tree Models
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42),

    # Boosting
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'AdaBoost': AdaBoostRegressor(random_state=42),
    'XGBoost': XGBRegressor(
        random_state=42,
        verbosity=0
    ),
    'LightGBM': LGBMRegressor(
        random_state=42,
        verbose=-1
    ),
    'CatBoost': CatBoostRegressor(
        random_state=42,
        verbose=0
    ),

    # Neural Network
    'MLP': MLPRegressor(
        random_state=42,
        max_iter=1000
    )
}

In [101]:
model_output = []

for model_name, model in model_dict.items():
    model_output.append(
        scorer(model_name, model)
    )

c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\preprocessing\_encoders

In [102]:
model_df = pd.DataFrame(
    model_output,
    columns=[
        'Model',
        'CV_R2',
        'Test_R2',
        'MAE',
        'RMSE'
    ]
)

model_df.sort_values(
    by='MAE',
    ascending=True
).reset_index(drop=True)

,Model,CV_R2,Test_R2,MAE,RMSE
0,Extra Trees,0.841765,0.860881,15.303475,24.374174
1,Random Forest,0.856391,0.860709,15.476140,25.547307
2,CatBoost,0.861206,0.857527,15.621209,24.962270
3,LightGBM,0.847061,0.850237,16.456430,27.898937
4,Gradient Boosting,0.845305,0.848571,16.499305,27.273486
5,SVR,0.850130,0.853114,16.638613,27.957609
6,XGBoost,0.849748,0.833963,16.887517,27.830439
7,Ridge,0.822849,0.845262,17.492583,28.061901
8,Linear Regression,0.812549,0.842769,17.624760,28.273033
9,MLP,0.756537,0.758690,18.414111,28.487779


### Target Encoding

In [121]:
!pip install category_encoders 

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 513.9 kB/s eta 0:00:18
   -- ------------------------------------- 0.5/9.5 MB 513.9 kB/s eta 0:00:18
   --- ------------------------------------ 0.8/9.5 MB 498.8 kB/s eta 0:00:18
   ---- ----------------------------------- 1.0/9.5 MB 647.4 kB/s eta 0:00:14
   ----- ---------------------------------- 1.3/9.5 MB 744.2 kB/s e


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [122]:
from category_encoders import TargetEncoder
from sklearn.pipeline import Pipeline

In [123]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            ['Area', 'Bathrooms', 'Balconies',
             'Current_Floor', 'Total_Floors', 'Bedrooms']
        ),
        (
            'cat',
            TargetEncoder(),
            ['Mapped_Area',
             'Property_Age',
             'Furnishing_Status',
             'Area_Type',
             'Property_Status',
             'Facing']
        )
    ]
)

In [125]:
model_dict = {

    # Linear Models
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),

    # Support Vector
    'SVR': SVR(),

    # Tree Models
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42),

    # Boosting
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'AdaBoost': AdaBoostRegressor(random_state=42),
    'XGBoost': XGBRegressor(
        random_state=42,
        verbosity=0
    ),
    'LightGBM': LGBMRegressor(
        random_state=42,
        verbose=-1
    ),
    'CatBoost': CatBoostRegressor(
        random_state=42,
        verbose=0
    ),

    # Neural Network
    'MLP': MLPRegressor(
        random_state=42,
        max_iter=1000
    )
}

In [126]:
model_output = []

for model_name, model in model_dict.items():
    model_output.append(
        scorer(model_name, model)
    )

c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\MEGH BAVARVA\Desktop\Da

In [127]:
model_df = pd.DataFrame(
    model_output,
    columns=[
        'Model',
        'CV_R2',
        'Test_R2',
        'MAE',
        'RMSE'
    ]
)

model_df.sort_values(
    by='MAE',
    ascending=True
).reset_index(drop=True)

,Model,CV_R2,Test_R2,MAE,RMSE
0,Extra Trees,0.858939,0.874088,14.642607,23.009805
1,Random Forest,0.862527,0.871517,15.250700,25.404373
2,CatBoost,0.863833,0.865048,15.524284,24.140503
3,Gradient Boosting,0.849464,0.851613,15.747552,25.337459
4,XGBoost,0.848949,0.861840,15.859983,25.519415
5,LightGBM,0.850161,0.853851,16.753530,28.528900
6,SVR,0.829232,0.838198,17.240262,28.887509
7,Decision Tree,0.776103,0.830688,17.326940,32.284660
8,MLP,0.822867,0.838224,18.030593,31.518143
9,Ridge,0.808339,0.832104,18.106143,30.242917


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            ['Area', 'Bathrooms', 'Balconies',
             'Current_Floor', 'Total_Floors', 'Bedrooms']
        ),
        (
            'target_enc',
            TargetEncoder(),
            ['Mapped_Area']
        ),
        (
            'ordinal_enc',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            ['Property_Age',
             'Furnishing_Status',
             'Area_Type',
             'Property_Status',
             'Facing']
        )
    ],
    remainder='passthrough'
)

In [130]:
model_dict = {

    # Linear Models
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),

    # Support Vector
    'SVR': SVR(),

    # Tree Models
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42),

    # Boosting
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'AdaBoost': AdaBoostRegressor(random_state=42),
    'XGBoost': XGBRegressor(
        random_state=42,
        verbosity=0
    ),
    'LightGBM': LGBMRegressor(
        random_state=42,
        verbose=-1
    ),
    'CatBoost': CatBoostRegressor(
        random_state=42,
        verbose=0
    ),

    # Neural Network
    'MLP': MLPRegressor(
        random_state=42,
        max_iter=1000
    )
}

In [131]:
model_output = []

for model_name, model in model_dict.items():
    model_output.append(
        scorer(model_name, model)
    )

c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\MEGH BAVARVA\Desktop\Da

In [132]:
model_df = pd.DataFrame(
    model_output,
    columns=[
        'Model',
        'CV_R2',
        'Test_R2',
        'MAE',
        'RMSE'
    ]
)

model_df.sort_values(
    by='MAE',
    ascending=True
).reset_index(drop=True)

,Model,CV_R2,Test_R2,MAE,RMSE
0,Extra Trees,0.858502,0.872602,14.795284,23.463178
1,Gradient Boosting,0.846240,0.854564,15.380796,24.305510
2,Random Forest,0.861249,0.866705,15.622391,25.816774
3,CatBoost,0.860615,0.858710,15.661591,24.633074
4,XGBoost,0.854449,0.856956,16.108233,26.042241
5,SVR,0.833898,0.845392,16.130627,26.848576
6,LightGBM,0.852047,0.852430,16.733727,28.489689
7,Decision Tree,0.765311,0.834603,17.309027,33.181413
8,MLP,0.801272,0.829069,17.326300,26.618388
9,Ridge,0.808431,0.834409,18.025936,29.911980


In [137]:
pip install hyperopt


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 1.1 MB/s eta 0:00:01
   ------------------- -------------------- 0.8/1.6 MB 1.3 MB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 1.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 3.9 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


### Hyperparameter Tuning

In [141]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV

In [142]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            ['Area', 'Bathrooms', 'Balconies',
             'Current_Floor', 'Total_Floors', 'Bedrooms']
        ),
        (
            'cat',
            TargetEncoder(),
            ['Mapped_Area',
             'Property_Age',
             'Furnishing_Status',
             'Area_Type',
             'Property_Status',
             'Facing']
        )
    ]
)

In [143]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', ExtraTreesRegressor(random_state=42))
])

In [155]:
param_dist = {
    'regressor__n_estimators': [200, 300,400, 500, 750, 1000,1500],
    'regressor__max_depth': [None, 10, 20, 30, 40,50],
    'regressor__min_samples_split': [2, 5, 10,15,20],
    'regressor__min_samples_leaf': [1, 2, 4,8],
    'regressor__max_features': ['sqrt', 'log2', None]
}

In [156]:
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=200,                
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42,
    verbose=2
)

In [157]:
random_search.fit(X, y_transformed)

Fitting 5 folds for each of 200 candidates, totalling 1000 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'regressor__max_depth': [None, 10, ...], 'regressor__max_features': ['sqrt', 'log2', ...], 'regressor__min_samples_leaf': [1, 2, ...], 'regressor__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",200
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Gu

In [158]:
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV MAE:")
print(-random_search.best_score_)

Best Parameters:
{'regressor__n_estimators': 1500, 'regressor__min_samples_split': 5, 'regressor__min_samples_leaf': 2, 'regressor__max_features': None, 'regressor__max_depth': 50}

Best CV MAE:
0.15815866606732495


In [159]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_transformed,
    test_size=0.3,
    random_state=42
)

best_model = random_search.best_estimator_

best_model.fit(X_train, y_train)

y_pred_log = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)

# Back to original price scale
y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test_actual, y_pred_actual)

rmse = np.sqrt(
    mean_squared_error(y_test_actual, y_pred_actual)
)

print(f"R² Score : {r2:.4f}")
print(f"MAE      : {mae:.4f}")
print(f"RMSE     : {rmse:.4f}")

R² Score : 0.8767
MAE      : 14.8610
RMSE     : 26.8479


regressor_n_estimators = 300
regressor_max_depth = 20
regressor_min_samples_split = 5
regressor_min_samples_leaf = 2
regressor__max_features = 'None'
MAE : 0.1579 

R2 score :0.8777
MAE:14.78
RMSE: 26.58

In [161]:
param_dist = {
    'regressor__n_estimators': [200, 250, 300, 350, 400, 450],
    'regressor__max_depth': [15, 18, 20, 22, 25],
    'regressor__min_samples_split': [2, 3, 4, 5, 6, 7, 8],
    'regressor__min_samples_leaf': [1, 2, 3, 4],
    'regressor__max_features': [None]
}

In [162]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_dist,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X, y_transformed)

Fitting 5 folds for each of 840 candidates, totalling 4200 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'regressor__max_depth': [15, 18, ...], 'regressor__max_features': [None], 'regressor__min_samples_leaf': [1, 2, ...], 'regressor__min_samples_split': [2, 3, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the mor

In [163]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV MAE:")
print(-grid_search.best_score_)

Best Parameters:
{'regressor__max_depth': 25, 'regressor__max_features': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 7, 'regressor__n_estimators': 350}

Best CV MAE:
0.1576183729080363


In [164]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_transformed,
    test_size=0.3,
    random_state=42
)

best_model = grid_search.best_estimator_

best_model.fit(X_train, y_train)

y_pred_log = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)

# Back to original price scale
y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test_actual, y_pred_actual)

rmse = np.sqrt(
    mean_squared_error(y_test_actual, y_pred_actual)
)

print(f"R² Score : {r2:.4f}")
print(f"MAE      : {mae:.4f}")
print(f"RMSE     : {rmse:.4f}")

R² Score : 0.8791
MAE      : 14.4203
RMSE     : 25.6870


In [170]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

In [171]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            ['Area', 'Bathrooms', 'Balconies',
             'Current_Floor', 'Total_Floors', 'Bedrooms']
        ),
        (
            'cat',
            TargetEncoder(),
            ['Mapped_Area',
             'Property_Age',
             'Furnishing_Status',
             'Area_Type',
             'Property_Status',
             'Facing']
        )
    ]
)

In [172]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

In [173]:
param_dist = {
    'regressor__n_estimators': [200, 300, 500, 750, 1000],
    'regressor__max_depth': [None, 10, 20, 30, 40],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': ['sqrt', 'log2', None],
    'regressor__bootstrap': [True, False]
}

In [174]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=200,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42,
    verbose=2
)

random_search.fit(X, y_transformed)

Fitting 5 folds for each of 200 candidates, totalling 1000 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'regressor__bootstrap': [True, False], 'regressor__max_depth': [None, 10, ...], 'regressor__max_features': ['sqrt', 'log2', ...], 'regressor__min_samples_leaf': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",200
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` 

In [175]:
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV MAE:")
print(-random_search.best_score_)

Best Parameters:
{'regressor__n_estimators': 200, 'regressor__min_samples_split': 5, 'regressor__min_samples_leaf': 2, 'regressor__max_features': 'sqrt', 'regressor__max_depth': 20, 'regressor__bootstrap': False}

Best CV MAE:
0.15727290674224662


In [176]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_transformed,
    test_size=0.2,
    random_state=42
)

best_model = random_search.best_estimator_

best_model.fit(X_train, y_train)

y_pred_log = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)

y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test_actual, y_pred_actual)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))

print(f"R² Score : {r2:.4f}")
print(f"MAE      : {mae:.4f}")
print(f"RMSE     : {rmse:.4f}")

R² Score : 0.8755
MAE      : 15.3039
RMSE     : 25.8979


In [177]:
param_dist = {
    'regressor__n_estimators': [100, 150, 200, 250, 300, 350],
    'regressor__max_depth': [15, 18, 20, 22, 25, 30],
    'regressor__min_samples_split': [3, 4, 5, 6, 7],
    'regressor__min_samples_leaf': [1, 2, 3],
    'regressor__max_features': ['sqrt'],
    'regressor__bootstrap': [False]
}

In [178]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'regressor__n_estimators': [150, 200, 250, 300],
    'regressor__max_depth': [18, 20, 22, 25],
    'regressor__min_samples_split': [4, 5, 6],
    'regressor__min_samples_leaf': [1, 2, 3],
    'regressor__max_features': ['sqrt'],
    'regressor__bootstrap': [False]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X, y_transformed)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'regressor__bootstrap': [False], 'regressor__max_depth': [18, 20, ...], 'regressor__max_features': ['sqrt'], 'regressor__min_samples_leaf': [1, 2, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages

In [179]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV MAE:")
print(-grid_search.best_score_)

Best Parameters:
{'regressor__bootstrap': False, 'regressor__max_depth': 25, 'regressor__max_features': 'sqrt', 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 6, 'regressor__n_estimators': 150}

Best CV MAE:
0.15663232813631006


In [180]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_transformed,
    test_size=0.2,
    random_state=42
)

best_model = grid_search.best_estimator_

best_model.fit(X_train, y_train)

y_pred_log = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)

y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test_actual, y_pred_actual)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))

print(f"R² Score : {r2:.4f}")
print(f"MAE      : {mae:.4f}")
print(f"RMSE     : {rmse:.4f}")

R² Score : 0.8782
MAE      : 15.1647
RMSE     : 25.4775


In [2]:
pip install hyperopt==0.2.7

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from hyperopt import hp

In [7]:
space = {
    'iterations': hp.quniform('iterations', 300, 1500, 100),
    'depth': hp.quniform('depth', 4, 10, 1),
    'learning_rate': hp.loguniform('learning_rate', -5, -1),
    'l2_leaf_reg': hp.uniform('l2_leaf_reg', 1, 15),
    'subsample': hp.uniform('subsample', 0.6, 1.0)
}

In [8]:
from hyperopt import STATUS_OK
from sklearn.model_selection import cross_val_score, KFold
from catboost import CatBoostRegressor
from sklearn.pipeline import Pipeline

def objective(params):

    model = CatBoostRegressor(
        iterations=int(params['iterations']),
        depth=int(params['depth']),
        learning_rate=params['learning_rate'],
        l2_leaf_reg=params['l2_leaf_reg'],
        subsample=params['subsample'],
        random_state=42,
        verbose=0
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    score = cross_val_score(
        pipeline,
        X,
        y_transformed,
        cv=cv,
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    ).mean()

    return {
        'loss': -score,
        'status': STATUS_OK
    }

In [23]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            ['Area', 'Bathrooms', 'Balconies',
             'Current_Floor', 'Total_Floors', 'Bedrooms']
        ),
        (
            'cat',
            TargetEncoder(),
            ['Mapped_Area',
             'Property_Age',
             'Furnishing_Status',
             'Area_Type',
             'Property_Status',
             'Facing']
        )
    ]
)

In [24]:
from hyperopt import fmin, tpe, Trials
import numpy as np

trials = Trials()

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=100,
    trials=trials,
    rstate=np.random.default_rng(42)
)

print(best)

100%|██████████| 100/100 [04:22<00:00,  2.63s/trial, best loss: 0.14474575557422426]
{'depth': np.float64(6.0), 'iterations': np.float64(1100.0), 'l2_leaf_reg': np.float64(1.9622562392303726), 'learning_rate': np.float64(0.02336501505908804), 'subsample': np.float64(0.7523264831222428)}


In [25]:
from catboost import CatBoostRegressor

final_cat = CatBoostRegressor(
    iterations=int(best['iterations']),
    depth=int(best['depth']),
    learning_rate=best['learning_rate'],
    l2_leaf_reg=best['l2_leaf_reg'],
    subsample=best['subsample'],
    random_state=42,
    verbose=0
)

In [26]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', final_cat)
])

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_transformed,
    test_size=0.2,
    random_state=42
)

In [28]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [29]:
y_pred_log = pipeline.predict(X_test)

In [30]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# R² on log-transformed target
r2 = r2_score(y_test, y_pred_log)

# Convert back to original price scale
y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test_actual, y_pred_actual)

rmse = np.sqrt(
    mean_squared_error(y_test_actual, y_pred_actual)
)

print(f"R² Score : {r2:.4f}")
print(f"MAE      : {mae:.4f}")
print(f"RMSE     : {rmse:.4f}")

R² Score : 0.8642
MAE      : 15.5532
RMSE     : 24.6776


In [31]:
from hyperopt import hp

space = {
    'iterations': hp.quniform('iterations', 900, 1400, 50),
    'depth': hp.quniform('depth', 5, 8, 1),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.05),
    'l2_leaf_reg': hp.uniform('l2_leaf_reg', 1, 5),
    'subsample': hp.uniform('subsample', 0.65, 0.90)
}

In [34]:
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=100,
    trials=Trials(),
    rstate=np.random.default_rng(42)
)

print(best)

100%|██████████| 100/100 [05:13<00:00,  3.14s/trial, best loss: 0.14433449490677858]
{'depth': np.float64(7.0), 'iterations': np.float64(1300.0), 'l2_leaf_reg': np.float64(2.993158342058905), 'learning_rate': np.float64(0.021734839362835878), 'subsample': np.float64(0.8333885959515775)}


In [35]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# R² on log-transformed target
r2 = r2_score(y_test, y_pred_log)

# Convert back to original price scale
y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test_actual, y_pred_actual)

rmse = np.sqrt(
    mean_squared_error(y_test_actual, y_pred_actual)
)

print(f"R² Score : {r2:.4f}")
print(f"MAE      : {mae:.4f}")
print(f"RMSE     : {rmse:.4f}")

R² Score : 0.8642
MAE      : 15.5532
RMSE     : 24.6776


In [36]:
space = {
    'n_estimators': hp.quniform('n_estimators', 200, 1500, 50),
    'max_depth': hp.quniform('max_depth', 3, 12, 1),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
    'gamma': hp.uniform('gamma', 0, 5),
    'reg_alpha': hp.uniform('reg_alpha', 0, 5),
    'reg_lambda': hp.uniform('reg_lambda', 0, 5)
}

In [37]:
def objective(params):

    model = XGBRegressor(
        n_estimators=int(params['n_estimators']),
        max_depth=int(params['max_depth']),
        learning_rate=params['learning_rate'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        min_child_weight=int(params['min_child_weight']),
        gamma=params['gamma'],
        reg_alpha=params['reg_alpha'],
        reg_lambda=params['reg_lambda'],
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    score = cross_val_score(
        pipeline,
        X,
        y_transformed,
        cv=cv,
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    ).mean()

    return {
        'loss': -score,
        'status': STATUS_OK
    }

In [38]:
trials = Trials()

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=100,
    trials=trials,
    rstate=np.random.default_rng(42)
)

print(best)

100%|██████████| 100/100 [00:56<00:00,  1.76trial/s, best loss: 0.14915090962260272]
{'colsample_bytree': np.float64(0.7240262144213081), 'gamma': np.float64(0.01465481841922026), 'learning_rate': np.float64(0.030240369675349123), 'max_depth': np.float64(6.0), 'min_child_weight': np.float64(9.0), 'n_estimators': np.float64(450.0), 'reg_alpha': np.float64(0.8080757350364929), 'reg_lambda': np.float64(1.3717976010091224), 'subsample': np.float64(0.6494271289828449)}


In [39]:
best_xgb = XGBRegressor(
    n_estimators=int(best['n_estimators']),
    max_depth=int(best['max_depth']),
    learning_rate=best['learning_rate'],
    subsample=best['subsample'],
    colsample_bytree=best['colsample_bytree'],
    min_child_weight=int(best['min_child_weight']),
    gamma=best['gamma'],
    reg_alpha=best['reg_alpha'],
    reg_lambda=best['reg_lambda'],
    random_state=42,
    n_jobs=-1
)

In [40]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', best_xgb)
])

pipeline.fit(X_train, y_train)

y_pred_log = pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred_log)

y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test_actual, y_pred_actual)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))

print(f"R² Score : {r2:.4f}")
print(f"MAE      : {mae:.4f}")
print(f"RMSE     : {rmse:.4f}")

R² Score : 0.8572
MAE      : 17.2129
RMSE     : 29.1229


| Model            | R²         | MAE         | RMSE        |
| ---------------- | ---------- | ----------- | ----------- |
|  Extra Trees   | **0.8791** | **14.4203** | 25.6870     |
|  Random Forest | 0.8782     | 15.1647     | 25.4775     |
|  CatBoost      | 0.8642     | 15.5532     | **24.6776** |
|  XGBoost      | 0.8572     | 17.2129     | 29.1229     |


In [ ]:
import joblib

# 2. Define the preprocessor and pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            ['Area', 'Bathrooms', 'Balconies',
             'Current_Floor', 'Total_Floors', 'Bedrooms']
        ),
        (
            'cat',
            TargetEncoder(),
            ['Mapped_Area',
             'Property_Age',
             'Furnishing_Status',
             'Area_Type',
             'Property_Status',
             'Facing']
        )
    ]
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', ExtraTreesRegressor(random_state=42))
])

# 3. Apply best hyperparameters found
best_params = {
    'regressor__max_depth': 25,
    'regressor__max_features': None,
    'regressor__min_samples_leaf': 1,
    'regressor__min_samples_split': 7,
    'regressor__n_estimators': 350
}
pipeline.set_params(**best_params)

# 4. Train the model on FULL data 
pipeline.fit(X, y_transformed)

# 5. Save the trained pipeline
joblib.dump(pipeline, "model_full_data.pkl")
print("Model trained on full data and saved as 'model_full_data.pkl'")

Model trained on full data and saved as 'model_full_data.pkl'
